# Oscillators and PLLs

This notebook covers how transmitters create stable RF. A voltage-controlled oscillator can wander; a phase-locked loop keeps it tied to a clean reference.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


## Locking a VCO to a Reference

The simplest intuition for a PLL is feedback: compare the VCO to a reference, measure the error, and nudge the VCO until the error shrinks.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))

def simulate_pll(loop_gain=0.08):
    ref = 10_000.0
    target = 8 * ref
    freqs = []
    errors = []
    vco = 72_000.0
    for _ in range(150):
        error = target - vco
        vco += loop_gain * error
        freqs.append(vco)
        errors.append(error)
    return np.array(freqs), np.array(errors)

def update_pll(loop_gain=0.08):
    freqs, errors = simulate_pll(loop_gain=loop_gain)
    axes[0].clear()
    axes[1].clear()
    axes[0].plot(freqs)
    axes[0].axhline(80_000, color="tab:red", linestyle="--", label="Target")
    axes[0].set_title("VCO Frequency vs Iteration")
    axes[0].set_xlabel("Iteration")
    axes[0].set_ylabel("Frequency (Hz)")
    axes[0].legend()
    axes[1].plot(errors, color="tab:orange")
    axes[1].set_title("Phase/Frequency Error")
    axes[1].set_xlabel("Iteration")
    axes[1].set_ylabel("Error (Hz)")
    fig.canvas.draw_idle()

controls = widgets.interactive(
    update_pll,
    loop_gain=float_slider(min_value=0.01, max_value=0.25, step=0.01, value=0.08, description="Loop gain"),
)
display(controls)


## Harmonics and Filtering

Power amplifiers and nonlinear stages create harmonics. Transmitters need output filtering so most of the power stays in the intended channel.

In [ ]:
fs = 200_000
t = np.arange(0, 0.01, 1 / fs)
fundamental = np.cos(2 * np.pi * 20_000 * t)
dirty = normalize(fundamental + 0.4 * np.cos(2 * np.pi * 40_000 * t) + 0.2 * np.cos(2 * np.pi * 60_000 * t))

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))

def update_harmonics(poles=4):
    axes[0].clear()
    axes[1].clear()
    sos = signal.butter(poles, 25_000, btype="low", fs=fs, output="sos")
    cleaned = signal.sosfilt(sos, dirty)
    plot_spectrum(dirty, fs=fs, ax=axes[0], title="Before output filter")
    plot_spectrum(cleaned, fs=fs, ax=axes[1], title="After output filter")
    for ax in axes:
        ax.set_xlim(0, 80_000)
        ax.set_ylim(-100, 5)
    fig.canvas.draw_idle()

controls = widgets.interactive(
    update_harmonics,
    poles=int_slider(min_value=2, max_value=10, step=2, value=4, description="Poles"),
)
display(controls)


## Key Takeaway

Transmitters need both frequency generation and spectral cleanup. Oscillators create the carrier, PLLs keep it where it belongs, and output filters keep harmonics from escaping.